# Qwen2.5-7B — VHE QLoRA with Dual DEV Decoding

Fine-tune Qwen2.5-7B for event extraction with QLoRA. Every epoch is evaluated on the same DEV set using both unconstrained greedy decoding and structurally constrained JSON decoding.

**Configurable inputs:**
- `model_path`: attached base model path.
- `train.jsonl` and `dev.jsonl`: dataset paths.
- `RESUME_FROM_CHECKPOINT`: start from scratch or resume training.

> Baseline and constrained best adapters are saved independently. TEST is never used for checkpoint selection.

# Kaggle resources
## Model
- Qwen2.5-7B: https://www.kaggle.com/models/qwen-lm/qwen2.5/Transformers/7b

## Dataset
- GENEVA: https://www.kaggle.com/datasets/md0612/geneva-adapted
- VHE: https://www.kaggle.com/datasets/md0612/vhe-adapted
- BKEE: https://www.kaggle.com/datasets/md0612/bkee-adapted
## Metrics
- https://www.kaggle.com/datasets/md0612/metrics

## 1. Install dependencies

Install 4-bit quantization support and an efficient JSON-schema decoding backend.

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" "lm-format-enforcer==0.11.3"

## 2. Imports

Import model, dataset, QLoRA, evaluation, and constrained-decoding utilities.

In [ ]:
from pathlib import Path
import json
import gc
import sys
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from transformers.trainer_utils import EvalLoopOutput

# lm-format-enforcer 0.11.3 imports this class from the legacy module.
# Transformers 5 keeps it in tokenization_utils_base, so expose a
# compatibility alias before importing the integration.
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
import transformers.tokenization_utils as tokenization_utils

if not hasattr(tokenization_utils, "PreTrainedTokenizerBase"):
    tokenization_utils.PreTrainedTokenizerBase = PreTrainedTokenizerBase

from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import (
    build_token_enforcer_tokenizer_data,
    build_transformers_prefix_allowed_tokens_fn,
)
from peft import (
    prepare_model_for_kbit_training,
    get_peft_model,
    LoraConfig,
)

## 3. Load tokenizer and base model

Load the Qwen2.5-7B base model attached to the Kaggle notebook in 4-bit mode for T4 GPUs.

In [ ]:
model_path = Path("/kaggle/input/qwen2.5/transformers/7b/1")
if not model_path.exists():
    raise FileNotFoundError(
        f"Qwen2.5-7B was not found at {model_path}. "
        "Attach the Qwen2.5-7B base model to this Kaggle notebook."
    )

# 4-bit NF4 quantization for QLoRA.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

# Split the quantized base model across the available GPUs.
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

# Prepare the quantized model for LoRA training.
model = prepare_model_for_kbit_training(model)

# Required for gradient checkpointing during training.
model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Model loaded!")
print("Device:", model.device)

## 4. Tokenize one example

Concatenate input and target tokens while computing loss only on the target and EOS token.

In [ ]:
def tokenize_example(example):
    input_text = example["input_text"]
    target_text = example["target_text"]

    input_tokens = tokenizer(
        input_text,
        add_special_tokens=True,
    )

    target_tokens = tokenizer(
        target_text,
        add_special_tokens=False,
    )

    input_ids = input_tokens["input_ids"]
    target_ids = target_tokens["input_ids"]

    input_ids = (
        input_tokens["input_ids"]
        + target_ids
        + [tokenizer.eos_token_id]
    )

    labels = (
        [-100] * len(input_tokens["input_ids"])
        + target_ids
        + [tokenizer.eos_token_id]
    )

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * len(input_ids),
    }

## 5. Format event-extraction data

Convert every sample into an `input_text` / JSON `target_text` pair.

In [ ]:
def format_example(example):
    text = example["text"]

    target = {
        "events": example["events"]
    }

    return {
        "input_text": text,
        "target_text": json.dumps(
            target,
            ensure_ascii=False,
            separators=(",", ":")
        )
    }

## 6. Data collator

Pad each batch dynamically and mask label padding with `-100`.

In [ ]:
def ee_data_collator(features):
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    labels = []

    for f in features:
        pad_length = max_length - len(f["input_ids"])

        input_ids.append(
            f["input_ids"] + [tokenizer.pad_token_id] * pad_length
        )

        attention_mask.append(
            f["attention_mask"] + [0] * pad_length
        )

        labels.append(
            f["labels"] + [-100] * pad_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

## 7. Configure and apply LoRA

Attach LoRA adapters to the attention and MLP projections.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)

## 8. Load and preprocess the dataset

Load the VHE TRAIN and DEV JSONL files, format them, and tokenize the loss-evaluation copies.

In [ ]:
# dataset = load_dataset(
#     "json",
#     data_files={
#         "train": "## Data path/train.jsonl",
#         "dev": "## Data path/dev.jsonl",
#     },
# )

dataset = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/md0612/vhe-adapted/train.jsonl",
        "dev": "/kaggle/input/datasets/md0612/vhe-adapted/dev.jsonl",
    },
)

# Raw datasets
train_raw = dataset["train"]
dev_raw = dataset["dev"]

# Formatted datasets
train_formatted = train_raw.map(format_example)
dev_formatted = dev_raw.map(format_example)

# Tokenized datasets for training
train_dataset = train_formatted.map(
    tokenize_example,
    remove_columns=train_formatted.column_names,
)

# Tokenized dataset for validation loss.
dev_dataset = dev_formatted.map(
    tokenize_example,
    remove_columns=dev_formatted.column_names,
)

# Raw/formatted dataset for generation evaluation
dev_generation_dataset = dev_formatted

## 9. DEV metrics

After every epoch, generate predictions on the full DEV set in both decoding modes and compute TI, TC, AI, AC, and average F1.

In [ ]:
# sys.path.insert(0, "## Metrics path")

sys.path.insert(0, "/kaggle/input/datasets/md0612/metrics")

from metrics.evaluator import Evaluator

metric_calculator = Evaluator()

## Dual-decoding trainer and checkpoint selector

The unconstrained branch uses deterministic greedy decoding. The constrained branch uses the same model and prompt but enforces the exact `{{"events": [...]}}` JSON structure. Event types, roles, text, and spans remain model decisions. Each branch keeps its own best DEV adapter.

In [ ]:
EVENT_OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "events": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "event_type": {"type": "string"},
                    "trigger": {
                        "type": "array",
                        "minItems": 1,
                        "items": {
                            "type": "object",
                            "properties": {
                                "text": {"type": "string"},
                                "span": {
                                    "type": "array",
                                    "items": {"type": "integer"},
                                    "minItems": 2,
                                    "maxItems": 2,
                                },
                            },
                            "required": ["text", "span"],
                            "additionalProperties": False,
                        },
                    },
                    "arguments": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "role": {"type": "string"},
                                "mentions": {
                                    "type": "array",
                                    "items": {
                                        "type": "object",
                                        "properties": {
                                            "text": {"type": "string"},
                                            "span": {
                                                "type": "array",
                                                "items": {"type": "integer"},
                                                "minItems": 2,
                                                "maxItems": 2,
                                            },
                                        },
                                        "required": ["text", "span"],
                                        "additionalProperties": False,
                                    },
                                },
                            },
                            "required": ["role", "mentions"],
                            "additionalProperties": False,
                        },
                    },
                },
                "required": ["event_type", "trigger", "arguments"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["events"],
    "additionalProperties": False,
}

RUN_ROOT = Path("/kaggle/working/ee_qwen2_5_7b_vhe")
PREDICTION_ROOT = RUN_ROOT / "dev_predictions"
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)


class DualDecodeGenerationTrainer(Trainer):
    def __init__(
        self,
        generation_dataset,
        tokenizer,
        evaluator,
        prediction_root,
        max_new_tokens=256,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.generation_dataset = generation_dataset
        self.tokenizer = tokenizer
        self.evaluator = evaluator
        self.prediction_root = Path(prediction_root)
        self.max_new_tokens = max_new_tokens

        tokenizer_data = build_token_enforcer_tokenizer_data(tokenizer)
        self.constrained_prefix_fn = (
            build_transformers_prefix_allowed_tokens_fn(
                tokenizer_data,
                JsonSchemaParser(EVENT_OUTPUT_SCHEMA),
            )
        )

    def _generate_mode(self, mode):
        predictions = []
        golds = []
        records = []
        constrained = mode == "constrained"

        with torch.inference_mode():
            for index, example in enumerate(self.generation_dataset):
                inputs = self.tokenizer(
                    example["input_text"],
                    return_tensors="pt",
                    add_special_tokens=True,
                )
                inputs = {
                    key: value.to(self.model.device)
                    for key, value in inputs.items()
                }
                input_length = inputs["input_ids"].shape[1]
                generation_kwargs = {
                    "max_new_tokens": self.max_new_tokens,
                    "do_sample": False,
                    "num_beams": 1,
                    "eos_token_id": self.tokenizer.eos_token_id,
                    "pad_token_id": (
                        self.tokenizer.pad_token_id
                        or self.tokenizer.eos_token_id
                    ),
                }
                if constrained:
                    generation_kwargs["prefix_allowed_tokens_fn"] = (
                        self.constrained_prefix_fn
                    )

                generated_ids = self.model.generate(
                    **inputs,
                    **generation_kwargs,
                )
                generated_tokens = generated_ids[:, input_length:]
                prediction = self.tokenizer.decode(
                    generated_tokens[0],
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False,
                )
                gold = {"events": example["events"]}
                predictions.append(prediction)
                golds.append(gold)
                records.append({
                    "index": index,
                    "text": example["text"],
                    "gold": gold,
                    "prediction": prediction,
                })

        epoch = int(round(self.state.epoch or 0))
        output_path = self.prediction_root / f"epoch-{epoch}-{mode}.jsonl"
        with output_path.open("w", encoding="utf-8") as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")

        results = self.evaluator.evaluate_on_all_sample(predictions, golds)
        valid_count = 0
        for prediction in predictions:
            try:
                parsed = json.loads(prediction)
                valid_count += int(
                    isinstance(parsed, dict)
                    and isinstance(parsed.get("events"), list)
                )
            except (json.JSONDecodeError, TypeError):
                pass
        results["valid_json_rate"] = valid_count / max(1, len(predictions))
        return results

    def evaluation_loop(
        self,
        dataloader,
        description,
        prediction_loss_only=None,
        ignore_keys=None,
        metric_key_prefix="eval",
    ):
        output = super().evaluation_loop(
            dataloader,
            description,
            prediction_loss_only,
            ignore_keys,
            metric_key_prefix,
        )
        was_training = self.model.training
        old_use_cache = self.model.config.use_cache
        self.model.eval()
        self.model.config.use_cache = True
        try:
            dual_results = {
                mode: self._generate_mode(mode)
                for mode in ("baseline", "constrained")
            }
        finally:
            self.model.config.use_cache = old_use_cache
            if was_training:
                self.model.train()

        metrics = dict(output.metrics)
        for mode, results in dual_results.items():
            metrics[f"{metric_key_prefix}_{mode}_avg_f1"] = results["avg_f1"]
            metrics[f"{metric_key_prefix}_{mode}_valid_json_rate"] = (
                results["valid_json_rate"]
            )
            for name in ("TI", "TC", "AI", "AC"):
                metrics[f"{metric_key_prefix}_{mode}_{name.lower()}_f1"] = (
                    results[name]["f1"]
                )
        self.dual_generation_results = dual_results
        return EvalLoopOutput(
            predictions=output.predictions,
            label_ids=output.label_ids,
            metrics=metrics,
            num_samples=output.num_samples,
        )


class DualBestEarlyStoppingCallback(TrainerCallback):
    def __init__(
        self,
        output_root,
        tokenizer,
        patience=3,
        threshold=0.002,
        grace_epochs=2,
    ):
        self.output_root = Path(output_root)
        self.tokenizer = tokenizer
        self.patience = patience
        self.threshold = threshold
        self.grace_epochs = grace_epochs
        self.best = {"baseline": float("-inf"), "constrained": float("-inf")}
        self.early_stop_reference = {
            "baseline": float("-inf"),
            "constrained": float("-inf"),
        }
        self.best_epoch = {"baseline": None, "constrained": None}
        self.stale_epochs = {"baseline": 0, "constrained": 0}

    def _save_state(self):
        payload = {
            "best_avg_f1": self.best,
            "best_epoch": self.best_epoch,
            "early_stop_reference": self.early_stop_reference,
            "stale_epochs": self.stale_epochs,
            "patience": self.patience,
            "threshold": self.threshold,
            "grace_epochs": self.grace_epochs,
        }
        self.output_root.mkdir(parents=True, exist_ok=True)
        (self.output_root / "dual_best_state.json").write_text(
            json.dumps(payload, indent=2),
            encoding="utf-8",
        )

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if not state.is_world_process_zero:
            return control

        epoch = int(round(state.epoch or 0))
        for mode in ("baseline", "constrained"):
            key = f"eval_{mode}_avg_f1"
            if metrics is None or key not in metrics:
                raise KeyError(f"Missing DEV selection metric: {key}")
            score = float(metrics[key])
            if score > self.best[mode]:
                self.best[mode] = score
                self.best_epoch[mode] = epoch
                best_dir = self.output_root / f"best-{mode}"
                model.save_pretrained(best_dir)
                self.tokenizer.save_pretrained(best_dir)

            significant_improvement = (
                score > self.early_stop_reference[mode] + self.threshold
            )
            if significant_improvement:
                self.early_stop_reference[mode] = score
                self.stale_epochs[mode] = 0
            elif epoch > self.grace_epochs:
                self.stale_epochs[mode] += 1
            else:
                self.stale_epochs[mode] = 0

        self._save_state()
        if (
            epoch > self.grace_epochs
            and all(value >= self.patience for value in self.stale_epochs.values())
        ):
            control.should_training_stop = True
        return control

## 10. TrainingArguments

Evaluate and save after every epoch. Dual best-model selection is handled by the callback below rather than by the single-metric Trainer selector.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(RUN_ROOT / "checkpoints"),

    num_train_epochs=10,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=4,

    load_best_model_at_end=False,

    gradient_checkpointing=True,
    fp16=True,
    report_to="none",
)


Epochs 1–2 are grace epochs: they are evaluated and can become best checkpoints, but they do not consume patience. Starting at epoch 3, training stops only when neither decoding branch improves by at least 0.002 for three consecutive DEV evaluations.

In [ ]:
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.002
EARLY_STOPPING_GRACE_EPOCHS = 2

## 11. Clear caches and inspect GPU memory

Clear Python/CUDA caches and print memory usage for every visible GPU.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated, "
        f"{torch.cuda.memory_reserved(i) / 1024**3:.2f} GB reserved"
    )

## 12. Initialize the trainer and verify DEV batching

Use the same full DEV set for validation loss, baseline generation, and constrained generation.

In [ ]:
dual_selection_callback = DualBestEarlyStoppingCallback(
    output_root=RUN_ROOT,
    tokenizer=tokenizer,
    patience=EARLY_STOPPING_PATIENCE,
    threshold=EARLY_STOPPING_THRESHOLD,
    grace_epochs=EARLY_STOPPING_GRACE_EPOCHS,
)

trainer = DualDecodeGenerationTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    generation_dataset=dev_generation_dataset,
    tokenizer=tokenizer,
    evaluator=metric_calculator,
    prediction_root=PREDICTION_ROOT,
    max_new_tokens=256,
    data_collator=ee_data_collator,
    callbacks=[dual_selection_callback],
)

batch = next(iter(trainer.get_eval_dataloader()))
print("DEV samples:", len(dev_dataset))
print("DEV batch shape:", tuple(batch["input_ids"].shape))

# Fail-fast generation smoke test before spending time on epoch 1.
smoke_example = dev_generation_dataset[0]
smoke_inputs = tokenizer(
    smoke_example["input_text"],
    return_tensors="pt",
    add_special_tokens=True,
)
smoke_inputs = {
    key: value.to(model.device)
    for key, value in smoke_inputs.items()
}
old_use_cache = model.config.use_cache
model.eval()
model.config.use_cache = True
try:
    with torch.inference_mode():
        model.generate(
            **smoke_inputs,
            max_new_tokens=4,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
        model.generate(
            **smoke_inputs,
            max_new_tokens=4,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            prefix_allowed_tokens_fn=trainer.constrained_prefix_fn,
        )
finally:
    model.config.use_cache = old_use_cache
    model.train()

print("PASS: baseline and constrained generation smoke test")

The run records separate DEV predictions and best adapters for both decoding modes.

In [ ]:
print(
    "Dual early stopping:",
    f"patience={EARLY_STOPPING_PATIENCE},",
    f"threshold={EARLY_STOPPING_THRESHOLD},",
    f"grace_epochs={EARLY_STOPPING_GRACE_EPOCHS}",
)

## 13. Start from scratch or resume

- Set `RESUME_FROM_CHECKPOINT = None` to train from scratch.
- Set it to a compatible Qwen2.5-7B QLoRA checkpoint path to resume.

Do not resume from a Qwen3-4B checkpoint because its base-model architecture is different.

In [ ]:
# Example: CHECKPOINT = "/kaggle/input/.../qwen2.5-7b-vhe/checkpoint-..."
# Use None for a fresh run, or set a compatible checkpoint path.
RESUME_FROM_CHECKPOINT = None

trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)

## 14. Report selected adapters and outputs

The callback has already saved the independently selected baseline and constrained adapters. Save the final training state separately for recovery.

In [ ]:
FINAL_MODEL_DIR = RUN_ROOT / "final-training-state"
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Best baseline epoch:", dual_selection_callback.best_epoch["baseline"])
print("Best baseline DEV avg_f1:", dual_selection_callback.best["baseline"])
print("Best constrained epoch:", dual_selection_callback.best_epoch["constrained"])
print("Best constrained DEV avg_f1:", dual_selection_callback.best["constrained"])
print("Best adapters and prediction files:", RUN_ROOT)